## Setting things up

### Import libraries
Add directory with the CIBUSmod modules to path to be able to import

In [1]:
import sys
import os
sys.path.insert(0, 'C:\\Users/jnka0003/Git repos/CIBUSmod')

Import CIBUSmod and packages for handling data and plotting

In [2]:
import CIBUSmod as cm

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


 -----------------------
|   Imported CIBUSmod   |
 -----------------------
commit : 6a426bc864c936c1067560c0d477019594a71a5b
branch : main
dirty  : True
remote : https://github.com/SLU-foodsystems/CIBUSmod
    


### Set up scenarios

In [3]:
# Create session
session = cm.Session(
    name = 'FORMAS_sens',
    data_path = 'C:\\Users/jnka0003/Git repos/CIBUSmod/data',
    data_path_scenarios = 'scenarios',
    data_path_output = 'output',
)

# Define scenarios
session.add_scenario(
    name = 'BL',
    scenario_workbooks = 'BASELINE', 
    modules = 'all',
    pars = 'all',
    years = 0
)
session.add_scenario(
    name = 'MAX_CUR + FIX_ANI',
    scenario_workbooks = ['COMMON'], 
    modules = 'all',
    pars = 'all',
    years = [100]
)
session.add_scenario(
    name = 'ALL + MORE_WOODED',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_WOODED'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + LOW_YIELDS',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_LOW_YIELDS'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + HIGH_SHARE_SNG',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_HIGH_SHARE_SNG'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + MORE_POT',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_POT'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + MORE_CROPLAND',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_CROPLAND'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)
session.add_scenario(
    name = 'ALL + SNG_OBJ_ALT2',
    scenario_workbooks = ['COMMON', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES', 'SENS_MORE_CROPLAND'],
    modules = 'all',
    pars = 'all',
    years = [70,100,110]
)


A scenario with the name 'BL' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'MAX_CUR + FIX_ANI' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_WOODED' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + LOW_YIELDS' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + HIGH_SHARE_SNG' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_POT' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + MORE_CROPLAND' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL + SNG_OBJ_ALT2' already exists use .update_scenario() or .remove_scenario() instead.


## Run scenarios (multi proc.)

In [4]:
# Import
from concurrent.futures import ProcessPoolExecutor, as_completed
from multi_proc import do_run

In [5]:
# Create list of scenarios to run
runs = [(s,y) for s,y in session.iterate('2026-04-17')]
runs = [(s,y) for s,y in session.iterate('MAX_CUR + FIX_ANI')]
runs

[('MAX_CUR + FIX_ANI', '100')]

In [6]:
%%time
# Do the multi-processing
with ProcessPoolExecutor(max_workers=8) as executor:
    
    futures = {executor.submit(do_run, session, scn_year) : scn_year for scn_year in runs}

    for future in as_completed(futures):
    
        scn, year = futures[future]
           
        try:
            t = future.result()
        except Exception as ee:
            print(f'(!!!) {scn}, {year} failed with the exception: {ee}')
        else:
            m = int(t/60)
            s = int(round(t - m*60))
            print(f'{scn}, {year} finished successfully in {m}min {s}s')
            
session.cache.clear()

MAX_CUR + FIX_ANI, 100 finished successfully in 3min 2s
CPU times: total: 31.2 ms
Wall time: 3min 15s
